In [1]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, coalesce, when, regexp_replace, lit, substring, first

#создаёт Spark-сессию с автоматической загрузкой JAR-зависимостей для работы с внешними системами
spark = SparkSession.builder \
    .appName("SparkExample") \
    .config("spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.3.4,"
        "com.amazonaws:aws-java-sdk-bundle:1.12.262,"
        "ru.yandex.clickhouse:clickhouse-jdbc:0.3.2,"
        "org.postgresql:postgresql:42.5.0,"
        "org.apache.spark:spark-sql-kafka-0-10_2.12:3.3.0",
        ) \
    .getOrCreate()
"""
SparkSession.builder	—	Строитель Spark-сессии
.appName("SparkExample")	"SparkExample"	Имя в Spark UI
.config("spark.jars.packages", ...)	Список JAR	Автозагрузка библиотек Maven
.getOrCreate()	—	Создать/вернуть существующую
JAR-пакеты (что подключается)
┌─────────────────────────────────────┐
│ hadoop-aws:3.3.4                    │ S3/Minio (AWS S3 API)
├─────────────────────────────────────┤
│ aws-java-sdk-bundle:1.12.262        │ AWS SDK (аутентификация S3)
├─────────────────────────────────────┤
│ clickhouse-jdbc:0.3.2               │ ClickHouse (чтение/запись)
├─────────────────────────────────────┤
│ postgresql:42.5.0                   │ PostgreSQL (JDBC)
└─────────────────────────────────────┘ spark-sql-kafka → Kafka стриминг

"""

#подключает Spark к MinIO (S3-совместимое хранилище) через протокол s3a.
hadoop_conf = spark._jsc.hadoopConfiguration()
hadoop_conf.set("fs.s3a.access.key", os.getenv("MINIO_ROOT_USER"))
hadoop_conf.set("fs.s3a.secret.key", os.getenv("MINIO_ROOT_PASSWORD"))
hadoop_conf.set("fs.s3a.endpoint", "http://minio:9000")
hadoop_conf.set("fs.s3a.connection.ssl.enabled", "false")
hadoop_conf.set("fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
hadoop_conf.set("fs.s3a.path.style.access", "true")
"""
| Код                              | Параметр            | Значение            | Назначение                |
| -------------------------------- | ------------------- | ------------------- | ------------------------- |
| spark._jsc.hadoopConfiguration() | —                   | Hadoop-конфиг Spark | Доступ к файловой системе |
| fs.s3a.access.key                | MINIO_ROOT_USER     | "minioadmin"        | Логин MinIO               |
| fs.s3a.secret.key                | MINIO_ROOT_PASSWORD | "minioadmin"        | Пароль MinIO              |
| fs.s3a.endpoint                  | "http://minio:9000" | Адрес MinIO         | Docker-сервис             |
| fs.s3a.connection.ssl.enabled    | "false"             | Без HTTPS           | Локальный MinIO           |
| fs.s3a.impl                      | S3AFileSystem       | Файловая система    | S3A-драйвер               |
| fs.s3a.path.style.access         | "true"              | Path-style          | bucket.host/object        |

Spark ───┐
         ├─> Hadoop S3A ──> http://minio:9000
spark.read.csv("s3a://dev/shushiato/file.parquet")
         └─> [MINIO_ROOT_USER:MINIO_ROOT_PASSWORD]

- Логин/пароль → аутентификация
- endpoint → адрес MinIO (НЕ AWS S3)
- ssl=false → http://minio:9000 (Docker)
- path.style.access=true → minio:9000/dev/file (не dev.minio:9000)

"""


print("Активные Spark сессии:", spark.sparkContext.uiWebUrl)


:: loading settings :: url = jar:file:/usr/local/lib/python3.11/dist-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
ru.yandex.clickhouse#clickhouse-jdbc added as a dependency
org.postgresql#postgresql added as a dependency
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-dedfa734-25a9-4de6-9eff-f6e0fd80e5a6;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
	found ru.yandex.clickhouse#clickhouse-jdbc;0.3.2 in central
	found com.clickhouse#clickhouse-http-client;0.3.2 in central
	found com.clickhouse#clickhouse-client;0.3.2 in central
	found org.lz4#lz4-java;1.8.0 in central
	found com.google.code.gson#gson;2.8.8 in central
	found org.apache.httpcomponents#httpclient;4.5

Активные Spark сессии: http://f9da0f1b6342:4040


In [ ]:
# ⬇️ Параметры подключения к PostgreSQL
db_user = os.getenv("POSTGRES_USER")
db_password = os.getenv("POSTGRES_PASSWORD")
jdbc_url = "jdbc:postgresql://postgres_source:5432/source"
properties = {
    "user": db_user,
    "password": db_password,
    "driver": "org.postgresql.Driver"
}

table_name = "public.shops"
shops = spark.read.jdbc(
    jdbc_url,
    table_name,
    properties=properties
)  \
    .select("st_id", "shop_name")

#shops.show()

+-----+-----------+
|st_id|  shop_name|
+-----+-----------+
|  842|      Lenta|
|  843|     Magnit|
|  844|       Spar|
|  845|Pyaterochka|
|  846|      Lenta|
|  847|      Diksi|
|  848|      Lenta|
|  849|   FixPrice|
|  850|     Magnit|
|  851|      Lenta|
+-----+-----------+



In [ ]:
table_name = "public.shop_timezone"
    
timezones = spark.read.jdbc(jdbc_url, table_name, properties=properties) \
    .select(col("plant").alias("st_id"), col("time_zone").alias("timezone")) \
    .filter(col("time_zone").isNotNull()) \
    .filter(col("st_id") > 0) \
    .distinct() \
    .groupBy("st_id") \
    .agg(first("timezone").alias("timezone")) \
    .orderBy("st_id")

  



# # Взять ПОСЛЕДНИЙ символ substring(col("plant"), -1, 3)
# timezones = timezones.withColumn("timezone", substring(col("timezone"), -1, 1))
# df = df.withColumn("tz_code", 
#    substring(col("time_zone"), -1, 1).cast("int")
#)

#timezones.show()



+-----+--------+
|st_id|timezone|
+-----+--------+
|  842|        |
|  843|   RUS04|
|  844|        |
|  845|        |
|  847|   RUS03|
|  848|   RUS08|
+-----+--------+



In [ ]:
#join
df = shops.join(timezones, "st_id", "left") \
    .select(
        col("st_id"),
        col("shop_name"),
        coalesce(
            # RUS04 → 4
            when(col("timezone") == "RUS04", lit(4)),
            # RUSXX → число XX
            when(col("timezone").rlike("^RUS\\d+$"), 
                 regexp_replace(col("timezone"), "RUS", "").cast("int")),
            # Нет TZ → 3
            lit(3)
        ).alias("tz_code")
        
    ) \
    .orderBy("st_id")
df.show()

+-----+-----------+-------+
|st_id|  shop_name|tz_code|
+-----+-----------+-------+
|  842|      Lenta|      3|
|  843|     Magnit|      4|
|  844|       Spar|      3|
|  845|Pyaterochka|      3|
|  846|      Lenta|      3|
|  847|      Diksi|      3|
|  848|      Lenta|      8|
|  849|   FixPrice|      3|
|  850|     Magnit|      3|
|  851|      Lenta|      3|
+-----+-----------+-------+



In [7]:
spark.stop()